# RAFT++: Rejection Sampling Fine-Tuning with GVM Dynamic Allocation

This notebook applies **RAFT++** (Rejection Sampling Fine-Tuning) with **GVM-RAFT**
dynamic completion allocation to improve STEM reasoning.

**Training pipeline stage:** 2 of 4 (GSPO (curriculum) → **RAFT++** → AdaSTaR → DPO)

**Target hardware:** Google Colab A100 40GB / 80GB

**Key features:**
- **GVM-RAFT dynamic allocation (arXiv 2504.11343):** Pilot round (4 completions) estimates
  per-problem difficulty, remaining budget allocated proportionally — hard problems get more attempts.
  2-4x speedup over fixed allocation.
- **Negative saving (arXiv 2505.24850):** Incorrect completions saved to JSONL for DPO stage reuse,
  eliminating redundant generation in the final pipeline stage.
- Domain-specific verification with verify_answers.py
- Domain balancing: oversample minority domains to prevent imbalance
- Up to 2 RAFT rounds with early stopping
- Fresh LoRA adapter on top of GSPO checkpoint
- No negative penalties: only learn from correct solutions

**Domains:** Mathematics, Physics, Chemistry, Biology, Computer Science

**References:**
- [GVM-RAFT / Minimalist Approach (arXiv 2504.11343)](https://arxiv.org/abs/2504.11343) — Dynamic allocation
- [Harnessing Negative Signals (arXiv 2505.24850)](https://arxiv.org/abs/2505.24850) — Reusing incorrect completions
- [ReST (arXiv 2308.08998)](https://arxiv.org/abs/2308.08998) — Reinforced Self-Training
- [STaR (arXiv 2203.14465)](https://arxiv.org/abs/2203.14465) — Self-Taught Reasoner

In [ ]:
# Disable gradient offloading BEFORE importing unsloth
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"

# Install dependencies — TRL >= 0.27.0 pinned for consistency across pipeline
!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy  # For verifiable reward functions

# Login to HuggingFace
from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Add Drive root to sys.path (training/ is at MyDrive level)
# ============================================================
import sys, os

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py", "training/scripts/sort_curriculum.py"]:
    print(f"  {'OK' if os.path.exists(os.path.join(DRIVE_ROOT, s)) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Configuration
# ============================================================

BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
GSPO_CHECKPOINT = "/content/drive/MyDrive/MITS/checkpoints/gspo_qwen3_4b/final_adapter"
GSPO_HF_REPO = "Siesher/mits-qwen3-4b-gspo"
OUTPUT_DIR = "/content/drive/MyDrive/MITS/checkpoints/raft_qwen3_4b"

# ---- A100 GPU preset ----
A100_VRAM_GB = 40  # Set to actual VRAM (40GB or 80GB)

# RAFT++ parameters
if A100_VRAM_GB >= 80:
    TOTAL_BUDGET_PER_PROBLEM = 8    # 4 pilot + up to 4 dynamic
    SFT_BATCH_SIZE = 4
    GEN_BATCH_SIZE = 10             # 10 × 4 seq = 40 parallel sequences
else:
    TOTAL_BUDGET_PER_PROBLEM = 8
    SFT_BATCH_SIZE = 2
    GEN_BATCH_SIZE = 6              # 6 × 4 seq = 24 parallel sequences via cuBLAS

MAX_ROUNDS = 2
MAX_COMPLETION = 512               # 512 is enough; shorter → faster; thinking fits in 300-400 tok
MAX_PROMPT_LENGTH = 512
GENERATION_TEMPERATURE = 0.9

# ---- GVM-RAFT (arXiv 2504.11343) ----
GVM_PILOT_SIZE = 4
GVM_MIN_ADDITIONAL = 1             # Minimum 1 extra completion

# ---- RAFT++ dataset size per round ----
# native_matmul() fix enables true parallel cuBLAS — 800 problems feasible on A100 40GB
RAFT_SAMPLE_SIZE = 800

# ---- Negative Saving (arXiv 2505.24850) ----
NEGATIVES_PATH = os.path.join(OUTPUT_DIR, "raft_negatives.jsonl") if 'OUTPUT_DIR' in dir() else ""

# SFT parameters
SFT_LR = 2e-5
SFT_EPOCHS = 1
SFT_GRAD_ACCUM = 4
SFT_MAX_SEQ_LENGTH = MAX_PROMPT_LENGTH + MAX_COMPLETION

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

DOMAINS = ["math", "physics", "chemistry", "biology", "cs"]
SYSTEM_PROMPT = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."

print(f"Hardware: A100 {A100_VRAM_GB}GB")
print(f"RAFT++: budget={TOTAL_BUDGET_PER_PROBLEM}/problem, gen_batch={GEN_BATCH_SIZE} ({GEN_BATCH_SIZE*GVM_PILOT_SIZE} seq/call)")
print(f"Dataset: {RAFT_SAMPLE_SIZE} problems/round, max_completion={MAX_COMPLETION} tokens")
print(f"SFT: lr={SFT_LR}, epochs={SFT_EPOCHS}, batch={SFT_BATCH_SIZE}")

In [ ]:
# ============================================================
# Mount Drive + resolve GSPO checkpoint
# ============================================================
import json
import os

DRIVE_MOUNTED = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_MOUNTED = True
    print("Google Drive mounted successfully")
except Exception as e:
    print(f"Drive mount failed ({e}), using local storage")
    OUTPUT_DIR = "/content/checkpoints/raft_qwen3_4b"

# Resolve GSPO checkpoint: Drive first, then download from HuggingFace
if os.path.exists(GSPO_CHECKPOINT):
    print(f"GSPO checkpoint found on Drive: {GSPO_CHECKPOINT}")
elif GSPO_HF_REPO:
    from huggingface_hub import snapshot_download
    GSPO_CHECKPOINT = snapshot_download(GSPO_HF_REPO)
    print(f"Downloaded GSPO adapter from HuggingFace to: {GSPO_CHECKPOINT}")
else:
    raise FileNotFoundError(f"GSPO checkpoint not found: {GSPO_CHECKPOINT}")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# ============================================================
# Load RL problems: Drive JSONL → HF "rl" config → HF "gspo" fallback
# ============================================================
import json
import random
from datasets import load_dataset
from collections import Counter, defaultdict

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

def load_rl_problems():
    """Load RL dataset with fallback chain."""
    if os.path.exists(RL_DATA_PATH):
        print(f"Loading RL data from Drive: {RL_DATA_PATH}")
        problems = []
        with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
            for line in f:
                problems.append(json.loads(line))
        print(f"Loaded {len(problems)} problems from Drive JSONL")
        return problems
    try:
        print("Trying HF dataset config 'rl'...")
        hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
        problems = [dict(r) for r in hf_ds["train"]]
        if "test" in hf_ds:
            problems += [dict(r) for r in hf_ds["test"]]
        print(f"Loaded {len(problems)} from HF 'rl' config")
        return problems
    except Exception:
        pass
    print("Falling back to HF 'gspo' config...")
    hf_ds = load_dataset("Siesher/mits-stem-training-data", "gspo")
    problems = [dict(r) for r in hf_ds["train"]] + [dict(r) for r in hf_ds["test"]]
    print(f"Loaded {len(problems)} from HF 'gspo' config (fallback)")
    return problems


problems = load_rl_problems()
verifiable_problems_pool = [
    p for p in problems
    if p.get("type", "verifiable") == "verifiable"
    and p.get("answer_type", "numeric") != "conceptual"
]
print(f"Total verifiable pool: {len(verifiable_problems_pool)}")


def sample_for_round(round_idx, n=None):
    """Stratified sample by domain — different 800 problems per round.

    Round 0: seed=42, Round 1: seed=43, etc.
    Across 2 rounds, covers up to 1600 unique problems.
    """
    if n is None:
        n = RAFT_SAMPLE_SIZE
    if len(verifiable_problems_pool) <= n:
        return verifiable_problems_pool[:]

    rng = random.Random(42 + round_idx)  # Different seed per round

    domain_groups = defaultdict(list)
    for p in verifiable_problems_pool:
        domain_groups[p.get("domain", "unknown")].append(p)

    per_domain = n // len(domain_groups)
    sampled = []
    for domain, probs in sorted(domain_groups.items()):
        sampled.extend(rng.sample(probs, min(per_domain, len(probs))))

    # Fill remainder if any domain was undersized
    if len(sampled) < n:
        already = {id(p) for p in sampled}
        leftover = [p for p in verifiable_problems_pool if id(p) not in already]
        rng.shuffle(leftover)
        sampled.extend(leftover[:n - len(sampled)])

    rng.shuffle(sampled)
    return sampled


# Preview round 0 distribution
_preview = sample_for_round(0)
print(f"RAFT++ per round: {len(_preview)} problems (stratified by domain)")
domain_counts = Counter(p.get("domain", "unknown") for p in _preview)
for domain, count in sorted(domain_counts.items()):
    print(f"  {domain}: {count}")

In [ ]:
# ============================================================
# Load model + GSPO adapter (auto-detect PEFT)
# ============================================================
import torch
from unsloth import FastLanguageModel

# Use from_pretrained with adapter repo — Unsloth reads adapter_config.json,
# loads base model, applies adapter + optimizations automatically.
# This avoids the 504 unexpected / 903 missing keys bug from manual load_state_dict.
_gspo_source = GSPO_HF_REPO if GSPO_HF_REPO else GSPO_CHECKPOINT
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=_gspo_source,
    max_seq_length=SFT_MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
)

if hasattr(model, 'peft_config'):
    _cfg = list(model.peft_config.values())[0]
    effective_r = _cfg.r
    gspo_alpha = _cfg.lora_alpha
    print(f"GSPO adapter loaded from {_gspo_source}: r={effective_r}, alpha={gspo_alpha}")
else:
    effective_r = LORA_R
    gspo_alpha = LORA_ALPHA
    print(f"WARNING: peft_config not found after loading {_gspo_source}")

model.gradient_checkpointing_enable()

if hasattr(model, 'hf_device_map'):
    model.hf_device_map = {'': 0}

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

In [ ]:
# ============================================================
# Verification functions
# ============================================================
import re

# Try importing from stem_rewards / verify_answers
_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify functions from training.scripts.verify_answers")
except ImportError:
    try:
        import sys
        sys.path.insert(0, "/content/drive/MyDrive")
        from training.scripts.verify_answers import verify, extract_answer
        _verify_imported = True
        print("Imported verify functions (via Drive path)")
    except ImportError:
        print("WARNING: verify_answers not found, using fallback")

if not _verify_imported:
    import sympy

    def extract_answer(text):
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
        if boxed:
            return boxed[-1].strip()
        numbers = re.findall(r'[-+]?\d*\.?\d+', text)
        return numbers[-1] if numbers else text.strip()

    def verify_simple(answer, truth, domain):
        if domain == "math":
            try:
                pred = sympy.sympify(answer)
                gold = sympy.sympify(truth)
                return sympy.simplify(pred - gold) == 0
            except Exception:
                return answer.strip() == str(truth).strip()
        elif domain == "physics":
            try:
                pred_num = float(re.findall(r'[-+]?\d*\.?\d+', answer)[0])
                gold_num = float(re.findall(r'[-+]?\d*\.?\d+', str(truth))[0])
                return abs(pred_num - gold_num) / max(abs(gold_num), 1e-10) < 0.05
            except (ValueError, IndexError):
                return answer.strip() == str(truth).strip()
        else:
            return answer.strip().lower() == str(truth).strip().lower()


def verify_completion(completion, problem):
    """Verify a single completion against its problem's ground truth."""
    answer = extract_answer(completion)
    truth = problem.get("ground_truth", problem.get("answer", ""))
    domain = problem.get("domain", "math")

    if _verify_imported:
        result = verify(
            answer=answer, truth=truth, domain=domain,
            question_type=problem.get("type", "calc"),
            test_cases=problem.get("test_cases"),
        )
        return result.correct
    else:
        return verify_simple(answer, truth, domain)


print("Verification functions ready")

In [ ]:
# ============================================================
# GVM-RAFT: Generate, Verify, Filter (arXiv 2504.11343)
# + Negative Saving (arXiv 2505.24850)
# ============================================================
import random
from collections import defaultdict
from contextlib import contextmanager


@contextmanager
def native_matmul():
    """Bypass Unsloth's matmul+resize bottleneck during inference.

    Unsloth pre-allocates `out` as [1, N, 2560] but the required shape
    is [N, 1, 2560]. Every single token × every layer triggers a CUDA
    resize+realloc (~500 ops per token). By ignoring `out`, cuBLAS
    allocates the correct shape directly — zero resizes, no overhead.

    Safe: calling code does `out = torch_matmul(X, W.t(), out=out)`,
    so `out` gets the new correctly-shaped tensor anyway.
    """
    try:
        import unsloth.kernels.utils as _unk
        _orig = _unk.torch_matmul
        _unk.torch_matmul = lambda A, B, out=None: torch.matmul(A, B)
        patched = True
    except Exception:
        patched = False
    try:
        yield
    finally:
        if patched:
            _unk.torch_matmul = _orig


def generate_and_filter(model, tokenizer, problems, n_completions=16, gen_batch_size=4):
    """GVM-RAFT with batched pilot generation and zero-resize cuBLAS matmul."""
    model.eval()
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    tokenizer.padding_side = "left"
    _pad_id = tokenizer.eos_token_id
    _eos_id = tokenizer.eos_token_id

    correct_pairs = []
    incorrect_pairs = []
    stats = {"total_generated": 0, "total_correct": 0, "pilot_solved": 0,
             "dynamic_solved": 0, "problems_attempted": len(problems)}

    for batch_start in range(0, len(problems), gen_batch_size):
        batch = problems[batch_start:batch_start + gen_batch_size]

        batch_prompts = []
        for problem in batch:
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": problem["prompt"]},
            ]
            batch_prompts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            ))

        inputs = tokenizer(
            batch_prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_PROMPT_LENGTH,
        ).to(model.device)
        padded_prompt_len = inputs["input_ids"].shape[1]

        # ---- Pilot round: zero-resize cuBLAS for batched generation ----
        with torch.inference_mode(), native_matmul():
            pilot_outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_COMPLETION,
                temperature=GENERATION_TEMPERATURE,
                do_sample=True,
                num_return_sequences=GVM_PILOT_SIZE,
                pad_token_id=_pad_id,
                eos_token_id=_eos_id,
            )
        stats["total_generated"] += len(batch) * GVM_PILOT_SIZE

        pilot_correct_counts = []
        for bi, problem in enumerate(batch):
            pilot_correct = 0
            for k in range(GVM_PILOT_SIZE):
                seq_idx = bi * GVM_PILOT_SIZE + k
                comp = tokenizer.decode(
                    pilot_outputs[seq_idx][padded_prompt_len:], skip_special_tokens=True
                )
                if verify_completion(comp, problem):
                    pilot_correct += 1
                    correct_pairs.append((batch_prompts[bi], comp, problem))
                    stats["total_correct"] += 1
                else:
                    incorrect_pairs.append((batch_prompts[bi], comp, problem))
            pilot_correct_counts.append(pilot_correct)
            if pilot_correct > 0:
                stats["pilot_solved"] += 1

        # ---- Dynamic allocation: per-problem ----
        tokenizer.padding_side = "right"
        for bi, (problem, pilot_correct) in enumerate(zip(batch, pilot_correct_counts)):
            pilot_pass_rate = pilot_correct / GVM_PILOT_SIZE
            if pilot_pass_rate >= 1.0:
                continue
            remaining_budget = n_completions - GVM_PILOT_SIZE
            if pilot_correct == 0:
                additional = max(GVM_MIN_ADDITIONAL, remaining_budget)
            else:
                additional = max(GVM_MIN_ADDITIONAL, int(remaining_budget * (1.0 - pilot_pass_rate)))
            additional = min(additional, remaining_budget)
            if additional <= 0:
                continue

            prob_inputs = tokenizer(
                batch_prompts[bi], return_tensors="pt",
                truncation=True, max_length=MAX_PROMPT_LENGTH,
            ).to(model.device)
            pp_len = prob_inputs["input_ids"].shape[1]

            with torch.inference_mode(), native_matmul():
                extra_outputs = model.generate(
                    **prob_inputs,
                    max_new_tokens=MAX_COMPLETION,
                    temperature=GENERATION_TEMPERATURE,
                    do_sample=True,
                    num_return_sequences=additional,
                    pad_token_id=_pad_id,
                    eos_token_id=_eos_id,
                )
            stats["total_generated"] += additional

            pilot_was_zero = (pilot_correct == 0)
            for j in range(extra_outputs.shape[0]):
                comp = tokenizer.decode(extra_outputs[j][pp_len:], skip_special_tokens=True)
                if verify_completion(comp, problem):
                    correct_pairs.append((batch_prompts[bi], comp, problem))
                    stats["total_correct"] += 1
                    if pilot_was_zero:
                        stats["dynamic_solved"] += 1
                        pilot_was_zero = False
                else:
                    incorrect_pairs.append((batch_prompts[bi], comp, problem))

        tokenizer.padding_side = "left"

        done = min(batch_start + gen_batch_size, len(problems))
        if done % 50 < gen_batch_size or done == len(problems):
            print(f"  {done}/{len(problems)}: {stats['total_correct']} correct / {stats['total_generated']} generated")

    tokenizer.padding_side = "right"
    model.train()
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    return correct_pairs, incorrect_pairs, stats


def domain_balance(pairs, oversample_factor=2):
    domain_groups = defaultdict(list)
    for pair in pairs:
        domain = pair[2].get("domain", "math")
        domain_groups[domain].append(pair)
    if not domain_groups:
        return pairs
    max_count = max(len(g) for g in domain_groups.values())
    balanced = []
    for domain, group in domain_groups.items():
        balanced.extend(group)
        if len(group) < max_count:
            shortage = min(max_count - len(group), len(group) * (oversample_factor - 1))
            balanced.extend(random.choices(group, k=shortage))
    random.shuffle(balanced)
    return balanced


def save_negatives(incorrect_pairs, path, round_num):
    if not path or not incorrect_pairs:
        return 0
    os.makedirs(os.path.dirname(path), exist_ok=True)
    count = 0
    with open(path, "a", encoding="utf-8") as f:
        for prompt_text, completion, problem in incorrect_pairs:
            record = {
                "prompt": problem["prompt"],
                "completion": completion,
                "domain": problem.get("domain", "math"),
                "ground_truth": problem.get("ground_truth", problem.get("answer", "")),
                "round": round_num,
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    print(f"  Saved {count} negatives to {path}")
    return count


print(f"GVM-RAFT ready: native_matmul() zero-resize patch enabled")
print(f"Pilot: {GEN_BATCH_SIZE} problems × {GVM_PILOT_SIZE} seq = {GEN_BATCH_SIZE*GVM_PILOT_SIZE} parallel sequences")

In [ ]:
# ============================================================
# RAFT++ Training Loop (with Colab disconnect recovery)
# ============================================================
from trl import SFTConfig, SFTTrainer
from datasets import Dataset


def find_latest_checkpoint(output_dir):
    """Find latest TRL checkpoint for resume after Colab disconnect."""
    if not os.path.isdir(output_dir):
        return None
    checkpoints = [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    path = os.path.join(output_dir, latest)
    print(f"  Found checkpoint: {path}")
    return path


# ---- Resume support ----
progress_path = os.path.join(OUTPUT_DIR, "progress.json")
start_round = 0
round_metrics = []
total_negatives_saved = 0

if os.path.exists(progress_path):
    with open(progress_path) as f:
        saved = json.load(f)
    round_metrics = saved.get("round_metrics", [])
    total_negatives_saved = saved.get("total_negatives_saved", 0)
    start_round = len(round_metrics)
    if start_round > 0:
        print(f"Resuming from round {start_round + 1}")
        last_adapter = os.path.join(OUTPUT_DIR, f"round_{start_round}_adapter", "adapter_model.safetensors")
        if os.path.exists(last_adapter):
            from safetensors.torch import load_file as _load_rd
            model.load_state_dict(_load_rd(last_adapter), strict=False)
            print(f"  Loaded adapter from round {start_round}")

# ---- Main RAFT++ loop ----
for round_idx in range(start_round, MAX_ROUNDS):
    print(f"\n{'='*60}")
    print(f"RAFT++ Round {round_idx + 1}/{MAX_ROUNDS}")
    print(f"{'='*60}")

    # Fresh sample each round: round 0→seed 42, round 1→seed 43
    # 2 rounds × 800 = up to 1600 unique problems across training
    round_problems = sample_for_round(round_idx)
    print(f"Sampled {len(round_problems)} problems (seed={42 + round_idx})")

    # Step 1: Generate and filter
    correct_pairs, incorrect_pairs, gen_stats = generate_and_filter(
        model, tokenizer, round_problems,
        n_completions=TOTAL_BUDGET_PER_PROBLEM,
        gen_batch_size=GEN_BATCH_SIZE,
    )
    total_negatives_saved += save_negatives(incorrect_pairs, NEGATIVES_PATH, round_idx + 1)
    print(f"Generated: {gen_stats['total_generated']}, Correct: {gen_stats['total_correct']}")
    print(f"Pass rate: {gen_stats['total_correct']/max(gen_stats['total_generated'],1)*100:.1f}%")

    if not correct_pairs:
        print("No correct completions found! Stopping RAFT++.")
        break

    # Step 2: Domain balance
    balanced_pairs = domain_balance(correct_pairs)
    print(f"After domain balancing: {len(balanced_pairs)} training examples")

    # Step 3: Format as SFT dataset
    sft_data = []
    for prompt_text, completion_text, problem in balanced_pairs:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem["prompt"]},
            {"role": "assistant", "content": completion_text},
        ]
        formatted = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        sft_data.append({"text": formatted})

    sft_dataset = Dataset.from_list(sft_data)
    print(f"SFT dataset: {len(sft_dataset)} examples")

    # Step 4: SFT on correct completions
    round_output = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}")
    sft_config = SFTConfig(
        output_dir=round_output,
        num_train_epochs=SFT_EPOCHS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        learning_rate=SFT_LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_seq_length=SFT_MAX_SEQ_LENGTH,
        bf16=True,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        optim="adamw_torch_fused",
        seed=42 + round_idx,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        processing_class=tokenizer,
    )

    sft_resume = find_latest_checkpoint(round_output)
    print(f"Starting SFT (round {round_idx + 1})...")
    result = trainer.train(resume_from_checkpoint=sft_resume)
    print(f"SFT loss: {result.training_loss:.4f}")

    del trainer
    torch.cuda.empty_cache()

    round_metrics.append({
        "round": round_idx + 1,
        "sft_loss": result.training_loss,
        "correct_pairs": len(correct_pairs),
        "balanced_pairs": len(balanced_pairs),
        "sample_seed": 42 + round_idx,
        "gen_stats": gen_stats,
    })

    round_adapter_path = os.path.join(OUTPUT_DIR, f"round_{round_idx + 1}_adapter")
    model.save_pretrained(round_adapter_path)
    tokenizer.save_pretrained(round_adapter_path)
    with open(progress_path, "w") as f:
        json.dump({
            "round_metrics": round_metrics,
            "total_negatives_saved": total_negatives_saved,
        }, f, indent=2, default=str)
    print(f"  Progress saved (round {round_idx + 1})")

print(f"\nRAFT++ complete! Ran {len(round_metrics)} rounds." if round_metrics else "No rounds completed")

In [ ]:
# ============================================================
# Save final adapter + metrics + push to HuggingFace
# ============================================================

final_adapter_path = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)
print(f"Final RAFT++ adapter saved to {final_adapter_path}")

# Save training metrics
eval_metrics = {
    "stage": "raft_plus",
    "pipeline_position": "2 of 4",
    "rounds": round_metrics,
    "total_negatives_saved": total_negatives_saved,
    "negatives_path": NEGATIVES_PATH,
}
eval_path = os.path.join(OUTPUT_DIR, "raft_eval_metrics.json")
with open(eval_path, "w") as f:
    json.dump(eval_metrics, f, indent=2, default=str)
print(f"Training metrics saved to {eval_path}")

# Save training config
config = {
    "stage": "raft_plus",
    "pipeline": "GSPO (curriculum) -> RAFT++ -> AdaSTaR -> DPO",
    "pipeline_position": "2 of 4",
    "base_model": BASE_MODEL,
    "gspo_source": _gspo_source,
    "hardware": f"A100 {A100_VRAM_GB}GB",
    "total_budget_per_problem": TOTAL_BUDGET_PER_PROBLEM,
    "max_rounds": MAX_ROUNDS,
    "actual_rounds": len(round_metrics),
    "gvm_pilot_size": GVM_PILOT_SIZE,
    "gvm_min_additional": GVM_MIN_ADDITIONAL,
    "gvm_dynamic_allocation": True,
    "negatives_saved": total_negatives_saved,
    "negatives_path": NEGATIVES_PATH,
    "sft_lr": SFT_LR,
    "sft_epochs": SFT_EPOCHS,
    "lora_r": effective_r,
    "lora_alpha": gspo_alpha,
    "lora_dropout": LORA_DROPOUT,
    "total_problems": len(verifiable_problems_pool),
    "domain_balancing": True,
    "references": [
        "GVM-RAFT arXiv:2504.11343",
        "Negative Reuse arXiv:2505.24850",
        "ReST arXiv:2308.08998",
    ],
}
config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Config saved to {config_path}")

# Push to HuggingFace
PUSH_TO_HUB = True
HF_REPO_ID = "Siesher/mits-qwen3-4b-raft"

if PUSH_TO_HUB:
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

print("\nDone! RAFT++ adapter ready for AdaSTaR (next stage).")
print(f"Negatives saved at {NEGATIVES_PATH} for DPO reuse.")